In [ ]:
!pip install transformers datasets accelerate -q
!pip install scikit-learn pandas lxml tqdm -q

In [ ]:
import zipfile
import os

zip_path = "/content/DDICorpus-2013.zip"
extract_path = "/content/DDICorpus"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted!")
print(os.listdir(extract_path))

Dataset extracted!
['DDICorpus']


In [ ]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm

data = []

base_path = "/content/DDICorpus"

for root, dirs, files in os.walk(base_path):
    for file in tqdm(files):
        if file.endswith(".xml"):
            file_path = os.path.join(root, file)

            tree = ET.parse(file_path)
            root_xml = tree.getroot()

            for sentence in root_xml.findall("sentence"):

                sentence_text = sentence.attrib.get("text", "")

                entities = {}

                for entity in sentence.findall("entity"):
                    entities[entity.attrib["id"]] = entity.attrib["text"]

                for pair in sentence.findall("pair"):

                    e1 = pair.attrib.get("e1")
                    e2 = pair.attrib.get("e2")

                    drug1 = entities.get(e1, "")
                    drug2 = entities.get(e2, "")

                    ddi = pair.attrib.get("ddi", "false")
                    interaction_type = pair.attrib.get("type", "none")

                    data.append({
                        "sentence": sentence_text,
                        "drug1": drug1,
                        "drug2": drug2,
                        "ddi": 1 if ddi == "true" else 0,
                        "label": interaction_type
                    })

df = pd.DataFrame(data)

print(df.head())
print(df.shape)

0it [00:00, ?it/s]
100%|██████████| 54/54 [00:00<00:00, 6968.14it/s]


                                            sentence  \
0  Therapeutic drug monitoring can avoid iatrogen...   
1  Gentamicin is an aminoglycoside antibiotic use...   
2  Due to its nephrotoxicity, gentamicin may caus...   
3  Therapeutic drug monitoring (TDM) of gentamici...   
4  Therapeutic drug monitoring (TDM) of gentamici...   

                           drug1                      drug2  ddi      label  
0  99mTc-methylene diphosphonate                 gentamicin    1     effect  
1                     Gentamicin  aminoglycoside antibiotic    0       none  
2                     gentamicin                  99mTc-MDP    1  mechanism  
3                     gentamicin                  99mTc-MDP    0       none  
4                     gentamicin        radiopharmaceutical    0       none  
(34449, 5)


In [ ]:
csv_path = "/content/ddi_dataset.csv"

df.to_csv(csv_path, index=False)

print("CSV saved:", csv_path)

CSV saved: /content/ddi_dataset.csv


In [ ]:
print(df["label"].value_counts())

label
none         29450
effect        2047
mechanism     1621
advise        1047
int            284
Name: count, dtype: int64


In [ ]:
df["text"] = (
    df["drug1"] +
    " [SEP] " +
    df["drug2"] +
    " [SEP] " +
    df["sentence"]
)

df = df[df["label"].notna()]
df.head()

,sentence,drug1,drug2,ddi,label,text
0,Therapeutic drug monitoring can avoid iatrogen...,99mTc-methylene diphosphonate,gentamicin,1,effect,99mTc-methylene diphosphonate [SEP] gentamicin...
1,Gentamicin is an aminoglycoside antibiotic use...,Gentamicin,aminoglycoside antibiotic,0,none,Gentamicin [SEP] aminoglycoside antibiotic [SE...
2,"Due to its nephrotoxicity, gentamicin may caus...",gentamicin,99mTc-MDP,1,mechanism,gentamicin [SEP] 99mTc-MDP [SEP] Due to its ne...
3,Therapeutic drug monitoring (TDM) of gentamici...,gentamicin,99mTc-MDP,0,none,gentamicin [SEP] 99mTc-MDP [SEP] Therapeutic d...
4,Therapeutic drug monitoring (TDM) of gentamici...,gentamicin,radiopharmaceutical,0,none,gentamicin [SEP] radiopharmaceutical [SEP] The...


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["target"] = le.fit_transform(df["label"])

print(le.classes_)

['advise' 'effect' 'int' 'mechanism' 'none']


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["target"],
    test_size=0.2,
    random_state=42
)

In [ ]:
from transformers import AutoTokenizer

model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

In [ ]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
import torch

class DDIDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels.iloc[idx]
        )

        return item

    def __len__(self):
        return len(self.labels)

train_dataset = DDIDataset(
    train_encodings,
    train_labels
)

test_dataset = DDIDataset(
    test_encodings,
    test_labels
)

In [ ]:
from transformers import AutoModelForSequenceClassification

num_labels = len(le.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical ar

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,0.250193,0.223359
2,0.192741,0.199113
3,0.148050,0.206756


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10335, training_loss=0.21820932248968633, metrics={'train_runtime': 4328.0869, 'train_samples_per_second': 19.102, 'train_steps_per_second': 2.388, 'total_flos': 1.0537005915615312e+16, 'train_loss': 0.21820932248968633, 'epoch': 3.0})

In [ ]:
predictions = trainer.predict(test_dataset)

preds = predictions.predictions.argmax(axis=1)

from sklearn.metrics import classification_report

print(classification_report(
    test_labels,
    preds,
    target_names=le.classes_
))

              precision    recall  f1-score   support

      advise       0.87      0.80      0.84       216
      effect       0.86      0.87      0.87       398
         int       0.78      0.83      0.80        52
   mechanism       0.78      0.72      0.75       332
        none       0.97      0.98      0.97      5892

    accuracy                           0.95      6890
   macro avg       0.85      0.84      0.85      6890
weighted avg       0.95      0.95      0.95      6890



In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model.to(device)


def predict_ddi(drug1, drug2, sentence):

    text = (
        f"{drug1} [SEP] "
        f"{drug2} [SEP] "
        f"{sentence}"
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # Move tensors to GPU
    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(
        outputs.logits,
        dim=1
    ).item()

    prediction = le.inverse_transform(
        [pred]
    )[0]

    return prediction


print(
    predict_ddi(
        "Aspirin",
        "Warfarin",
        "Aspirin may increase bleeding risk with warfarin"
    )
)

effect


phase 2

In [ ]:
from itertools import combinations


def generate_drug_pairs(drug_list):

    drug_pairs = list(
        combinations(drug_list, 2)
    )

    return drug_pairs


drugs = [
    "Metformin",
    "Aspirin",
    "Ibuprofen"
]

pairs = generate_drug_pairs(drugs)

print(pairs)

[('Metformin', 'Aspirin'), ('Metformin', 'Ibuprofen'), ('Aspirin', 'Ibuprofen')]


In [ ]:
def predict_multi_drug_interactions(
    drug_list
):

    pairs = generate_drug_pairs(
        drug_list
    )

    results = []

    for drug1, drug2 in pairs:

        sentence = (
            f"{drug1} may interact "
            f"with {drug2}"
        )

        prediction = predict_ddi(
            drug1,
            drug2,
            sentence
        )

        results.append({
            "drug1": drug1,
            "drug2": drug2,
            "interaction": prediction
        })

    return results

In [ ]:
drugs = [
    "Metformin",
    "Aspirin",
    "Ibuprofen"
]

results = predict_multi_drug_interactions(
    drugs
)

for r in results:
    print(r)

{'drug1': 'Metformin', 'drug2': 'Aspirin', 'interaction': 'int'}
{'drug1': 'Metformin', 'drug2': 'Ibuprofen', 'interaction': 'int'}
{'drug1': 'Aspirin', 'drug2': 'Ibuprofen', 'interaction': 'int'}


phase 3

In [ ]:
!pip install biopython sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.0 MB/s eta 0:00:00


In [ ]:
from Bio import Entrez
import re

Entrez.email = "tharuneshvar1356@gmail.com"


def search_pubmed(query, max_results=10):

    handle = Entrez.esearch(
        db="pubmed",
        term=query,
        retmax=max_results,
        sort="relevance"
    )

    record = Entrez.read(handle)

    return record["IdList"]


def fetch_pubmed_abstracts(pmid_list):

    ids = ",".join(pmid_list)

    handle = Entrez.efetch(
        db="pubmed",
        id=ids,
        rettype="abstract",
        retmode="xml"
    )

    records = Entrez.read(handle)

    results = []

    for article in records["PubmedArticle"]:

        try:
            article_data = article[
                "MedlineCitation"
            ]["Article"]

            title = article_data.get(
                "ArticleTitle", ""
            )

            abstract = ""

            if "Abstract" in article_data:
                abstract = " ".join(
                    article_data["Abstract"]
                    ["AbstractText"]
                )

            pmid = str(
                article[
                    "MedlineCitation"
                ]["PMID"]
            )

            results.append({
                "pmid": pmid,
                "title": title,
                "abstract": abstract
            })

        except:
            continue

    return results


def retrieve_drug_evidence(
    drug1,
    drug2
):

    query = f'''
    (
      "{drug1}"[Title/Abstract]
      AND
      "{drug2}"[Title/Abstract]
    )
    AND
    (
      "drug interactions"[MeSH]
      OR interaction
      OR contraindication
      OR toxicity
      OR adverse
      OR bleeding
      OR safety
      OR coadministration
    )
    '''

    pmids = search_pubmed(
        query,
        max_results=10
    )

    papers = fetch_pubmed_abstracts(
        pmids
    )

    # Filter irrelevant papers
    bad_keywords = [
        "adsorption",
        "quantum",
        "spectroscopy",
        "molecular docking",
        "physicochemical",
        "nanoparticle",
        "covalent framework",
        "genetic",
        "simulation"
    ]

    filtered = []

    for p in papers:

        text = (
            p["title"] +
            " " +
            p["abstract"]
        ).lower()

        if not any(
            bad in text
            for bad in bad_keywords
        ):
            filtered.append(p)

    return filtered[:3]

In [ ]:
def analyze_drug_pair(
    drug1,
    drug2
):
    sentence = (
        f"{drug1} may interact "
        f"with {drug2}"
    )

    interaction = predict_ddi(
        drug1,
        drug2,
        sentence
    )

    ddi_examples = retrieve_ddi_examples(
        drug1,
        drug2
    )

    pubmed_evidence = retrieve_drug_evidence(
        drug1,
        drug2
    )

    return {
        "drug1": drug1,
        "drug2": drug2,
        "interaction": interaction,
        "ddi_examples":
            ddi_examples,
        "pubmed":
            pubmed_evidence
    }

In [ ]:
result = analyze_drug_pair(
    "Aspirin",
    "Ibuprofen"
)

print("Drug Pair:")
print(result["drug1"], "↔",
      result["drug2"])

print("\nInteraction:")
print(result["interaction"])

print("\nDDI Examples:")
print(result["ddi_examples"])

print("\nPubMed Evidence:")

for paper in result["pubmed"]:

    print("\nPMID:",
          paper["pmid"])

    print("TITLE:",
          paper["title"])

    print("ABSTRACT:")
    print(
        paper["abstract"][:500]
    )

    print("="*80)

Drug Pair:
Aspirin ↔ Ibuprofen

Interaction:
int

DDI Examples:
                                                sentence   label
19001  Aspirin: Animal studies wshow that aspirin giv...  effect

PubMed Evidence:

PMID: 39466269
TITLE: Peptic Ulcer Disease: A Review.
ABSTRACT:
In the US, peptic ulcer disease affects 1% of the population and approximately 54 000 patients are admitted to the hospital annually for bleeding peptic ulcers. Approximately 10% of patients presenting with upper abdominal pain in a primary care setting have a peptic ulcer as the cause of their symptoms. The principal causes of peptic ulcer disease are Helicobacter pylori infection, which affects approximately 42% of patients with peptic ulcer disease, and aspirin or nonsteroidal anti-inflammato

PMID: 19949916
TITLE: Ibuprofen: pharmacology, efficacy and safety.
ABSTRACT:
This review attempts to bring together information from a large number of recent studies on the clinical uses, safety and pharmacological prope

phase 4

In [ ]:
!pip install sentence-transformers faiss-cpu -q

In [ ]:
ddi_corpus = df[
    df["label"] != "none"
][["sentence", "label"]]

ddi_corpus = ddi_corpus.drop_duplicates()

ddi_corpus.head()

,sentence,label
0,Therapeutic drug monitoring can avoid iatrogen...,effect
2,"Due to its nephrotoxicity, gentamicin may caus...",mechanism
11,Injection of estradiol 5 min before a nonletha...,effect
14,The serum estrogen concentrations of estradiol...,mechanism
20,Exogenous estradiol also appeared to influence...,effect


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.47k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/669k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
corpus_sentences = ddi_corpus[
    "sentence"
].tolist()

corpus_embeddings = (
    embedding_model.encode(
        corpus_sentences,
        show_progress_bar=True
    )
)

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

In [ ]:
import faiss
import numpy as np

dimension = corpus_embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(corpus_embeddings)
)

print(
    "FAISS index ready!"
)

FAISS index ready!


In [ ]:
def retrieve_semantic_evidence(
    query,
    drug1=None,
    drug2=None,
    top_k=3
):

    # STEP 1: Exact pair match from DDI dataset
    exact_matches = df[
        (
            (
                df["drug1"]
                .str.lower()
                == drug1.lower()
            )
            &
            (
                df["drug2"]
                .str.lower()
                == drug2.lower()
            )
        )
        |
        (
            (
                df["drug1"]
                .str.lower()
                == drug2.lower()
            )
            &
            (
                df["drug2"]
                .str.lower()
                == drug1.lower()
            )
        )
    ]

    # Remove weak labels
    exact_matches = exact_matches[
        exact_matches["label"] != "none"
    ]

    # Remove duplicates
    exact_matches = (
        exact_matches
        .drop_duplicates(
            subset=["sentence"]
        )
    )

    # If exact match exists → return it
    if len(exact_matches) > 0:

        results = []

        for _, row in (
            exact_matches
            .head(top_k)
            .iterrows()
        ):

            results.append({
                "sentence":
                    row["sentence"],
                "label":
                    row["label"]
            })

        return results

    # STEP 2: FAISS fallback
    query_embedding = (
        embedding_model.encode([query])
    )

    distances, indices = index.search(
        np.array(query_embedding),
        top_k * 10
    )

    retrieved = []

    for idx in indices[0]:

        row = ddi_corpus.iloc[idx]

        sentence = (
            row["sentence"]
            .lower()
        )

        if (
            drug1.lower()
            in sentence
            and
            drug2.lower()
            in sentence
        ):

            retrieved.append({
                "sentence":
                    row["sentence"],
                "label":
                    row["label"]
            })

        if len(retrieved) >= top_k:
            break

    return retrieved

In [ ]:
user_input = "Warfarin, Aspirin"

drugs = [
    d.strip()
    for d in user_input.split(",")
]

drug1 = drugs[0]
drug2 = drugs[1]

query = (
    f"Interaction between "
    f"{drug1} and {drug2}"
)

evidence = retrieve_semantic_evidence(
    query,
    drug1=drug1,
    drug2=drug2
)

print(evidence)

[]


phase 5 llm

In [ ]:
!pip install transformers accelerate sentencepiece -q

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
import re


def clean_text(text):

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    text = text.replace(
        "wshow",
        "show"
    )

    text = text.replace(
        "nonsteroidal anti-inflammatory agents",
        "NSAIDs"
    )

    return text.strip()

In [ ]:
def get_risk_level(
    interaction,
    evidence_found=True
):

    # If no evidence → uncertain
    if not evidence_found:
        return "Unknown"

    risk_map = {

        "mechanism":
        "High",

        "effect":
        "Moderate",

        "advice":
        "High",

        "int":
        "Moderate",

        "none":
        "Low"
    }

    return risk_map.get(
        interaction,
        "Unknown"
    )

In [ ]:
def generate_clinical_explanation(
    drug1,
    drug2,
    interaction,
    evidence
):

    label_map = {

        "effect":
        "Effect Interaction",

        "mechanism":
        "Mechanism Interaction",

        "advice":
        "Clinical Advice Interaction",

        "int":
        "General Drug Interaction",

        "none":
        "No Significant Interaction"
    }

    interaction_summary = {

        "effect":
        "The effectiveness of one or both drugs may change.",

        "mechanism":
        "A biological or pharmacological interaction may occur.",

        "advice":
        "Medical supervision is recommended when taking these drugs together.",

        "int":
        "A potential interaction between the drugs has been detected.",

        "none":
        "No major interaction was detected."
    }

    clinical_map = {

        "effect":
        "Drug effectiveness may decrease or increase when used together.",

        "mechanism":
        "The drugs may interact biologically and alter metabolism or absorption.",

        "advice":
        "Medical supervision is advised when combining these medications.",

        "int":
        "Potential interaction detected. Monitoring for adverse effects is recommended.",

        "none":
        "No major clinical concern detected."
    }

    if evidence:
      evidence_text = clean_text(
        evidence[0]["sentence"]
    )

    else:
      evidence_text = (
        f"No direct DDI corpus evidence "
        f"was found for {drug1} "
        f"and {drug2}. "
        f"Prediction is based on "
        f"model inference."
    )

    explanation = f"""
Drug Pair:
{drug1} ↔ {drug2}

Interaction Type:
{label_map.get(interaction)}

Risk Level:
{get_risk_level(interaction)}

Interaction Summary:
{interaction_summary.get(interaction)}

Scientific Evidence:
{evidence_text}

Clinical Concern:
{clinical_map.get(interaction)}
"""

    return explanation

In [ ]:
from itertools import combinations


user_input = input(
    "Enter medicines separated by comma: "
)

drug_list = [
    d.strip()
    for d in user_input.split(",")
]

pairs = list(
    combinations(drug_list, 2)
)

print("\nAnalyzing Drug Interactions...\n")

for drug1, drug2 in pairs:

    print("=" * 70)

    query = (
        f"Interaction between "
        f"{drug1} and {drug2}"
    )

    # Retrieve evidence
    evidence = (
        retrieve_semantic_evidence(
            query,
            drug1=drug1,
            drug2=drug2
        )
    )

    # Base prediction
    predicted_interaction = (
        predict_ddi(
            drug1,
            drug2,
            f"{drug1} may interact "
            f"with {drug2}"
        )
    )

    interaction = (
        predicted_interaction
    )

    # Normalize names
    drug1_clean = (
        drug1.strip().lower()
    )

    drug2_clean = (
        drug2.strip().lower()
    )

    # High-risk overrides
    high_risk_pairs = {

        frozenset([
            "sildenafil",
            "nitroglycerin"
        ]): "advice",

        frozenset([
            "warfarin",
            "aspirin"
        ]): "effect",

        frozenset([
            "clopidogrel",
            "omeprazole"
        ]): "mechanism",

        frozenset([
            "simvastatin",
            "clarithromycin"
        ]): "mechanism"
    }

    pair_key = frozenset([
        drug1_clean,
        drug2_clean
    ])

    # Rule override
    if pair_key in high_risk_pairs:

        interaction = (
            high_risk_pairs[pair_key]
        )

    # Evidence-aware correction
    elif evidence:

        evidence_text = (
            evidence[0]["sentence"]
            .lower()
        )

        if any(
            word in evidence_text
            for word in [
                "cyp",
                "metabolism",
                "enzyme",
                "inhibition",
                "absorption",
                "rhabdomyolysis",
                "toxicity",
                "metabolite"
            ]
        ):

            interaction = (
                "mechanism"
            )

        elif any(
            word in evidence_text
            for word in [
                "increase",
                "decrease",
                "reduced",
                "effectiveness",
                "activity"
            ]
        ):

            interaction = (
                "effect"
            )

        elif any(
            word in evidence_text
            for word in [
                "monitor",
                "avoid",
                "contraindicated",
                "caution",
                "blood pressure",
                "hypotension",
                "fatal",
                "dangerous"
            ]
        ):

            interaction = (
                "advice"
            )

    # Generate response
    response = (
        generate_clinical_explanation(
            drug1,
            drug2,
            interaction,
            evidence
        )
    )

    print(response)

Enter medicines separated by comma: paracetamol ,cetirizine 

Analyzing Drug Interactions...



NameError: name 'retrieve_semantic_evidence' is not defined

In [1]:
!git --version


git version 2.34.1


In [7]:
!pwd
!ls

/content
Drug-Conflict-Intelligence-RAG	sample_data


cp: cannot stat '/content/Drug_Conflict_Intelligence_RAG.ipynb': No such file or directory
